In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

In [2]:
# imports

from pprint import pprint

import torch

from enviroment_bj import (
    BlackjackConfig,
    BlackjackEnvironment,
    ObservationConfig,
    StartStateConfig,
    ACTION_ORDER,
)
from model.agents import (
    FeedForwardDoubleDQN,
    RecurrentDoubleDQN,
    DuelingRecurrentDoubleDQN,
)


In [3]:
# helpers para crear mesas

def make_env(profile, shoe=None, seed=11, start_state=None, **config_overrides):
    observation = ObservationConfig.for_profile(profile)
    config = BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=observation,
        **config_overrides,
    )
    env = BlackjackEnvironment(config=config, seed=seed, start_state=start_state)
    if shoe is not None:
        env.load_shoe(shoe, total_cards=len(shoe))
    return env


def print_response_summary(response, title="response"):
    print("=" * 90)
    print(title)
    print("observation profile:", response["observation"].get("profile"))
    print("observation keys:", sorted(response["observation"].keys()))
    print("table_rules keys:", sorted(response["table_rules"].keys()))
    print("action_mask:", response["action_mask"])
    print("ACTION_ORDER:", ACTION_ORDER)


In [4]:
# mesa para el modelo feedforward
# Perfil mínimo, pensado para baseline simple

env_ff = make_env(
    "minimal_basic_strategy",
    shoe=["10", "6", "7", "10"],
    seed=11,
)

response_ff = env_ff.reset()
print_response_summary(response_ff, "feedforward env / reset")


feedforward env / reset
observation profile: minimal_basic_strategy
observation keys: ['current_hand_is_soft', 'current_hand_total', 'dealer_upcard', 'dealer_upcard_value', 'hand_context', 'insurance_context', 'mode', 'profile']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'resplit_aces_allowed', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 0, 1, 0]
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')


In [5]:
#  instanciar FeedForwardDoubleDQN en CPU

device = torch.device("cpu")

ff_model = FeedForwardDoubleDQN.from_profile("minimal_basic_strategy").to(device)
ff_model.eval()

print(ff_model)
print("state_dim:", ff_model.state_dim)
print("num_actions:", ff_model.num_actions)
print("device:", next(ff_model.parameters()).device)


FeedForwardDoubleDQN(
  (encoder): BlackjackObservationEncoder(
    (modules_by_name): ModuleDict(
      (hand): HandFeatureEncoder()
      (hand_context): HandContextEncoder()
      (insurance): InsuranceContextEncoder()
      (rules): RuleEncoder()
    )
  )
  (backbone): Sequential(
    (0): Linear(in_features=40, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (q_head): Linear(in_features=256, out_features=6, bias=True)
)
state_dim: 40
num_actions: 6
device: cpu


In [6]:
# forward del FeedForwardDoubleDQN

with torch.no_grad():
    ff_out = ff_model(response_ff)

print("q_values shape:", tuple(ff_out["q_values"].shape))
print("masked_q_values shape:", tuple(ff_out["masked_q_values"].shape))
print("state_vector shape:", tuple(ff_out["state_vector"].shape))
print("action_mask shape:", tuple(ff_out["action_mask"].shape))
print("backbone_output shape:", tuple(ff_out["backbone_output"].shape))
print("metadata:")
pprint(ff_out["metadata"])

print("\nq_values:")
print(ff_out["q_values"])

print("\nmasked_q_values:")
print(ff_out["masked_q_values"])


q_values shape: (1, 6)
masked_q_values shape: (1, 6)
state_vector shape: (1, 40)
action_mask shape: (1, 6)
backbone_output shape: (1, 256)
metadata:
{'architecture': 'feedforward',
 'batch_shape': (1, 6),
 'encoder_profile': 'minimal_basic_strategy',
 'module_dims': {'hand': 16, 'hand_context': 5, 'insurance': 2, 'rules': 17},
 'module_slices': {'hand': (0, 16),
                   'hand_context': (16, 21),
                   'insurance': (21, 23),
                   'rules': (23, 40)},
 'observation_mode': 'basic_strategy',
 'observation_profile': 'minimal_basic_strategy',
 'profile': 'minimal_basic_strategy',
 'state_dim': 40}

q_values:
tensor([[-0.0086, -0.0247,  0.0172,  0.0725,  0.0824, -0.0080]])

masked_q_values:
tensor([[-8.6241e-03, -2.4745e-02,  1.7151e-02, -3.4028e+38,  8.2387e-02,
         -3.4028e+38]])


In [7]:
#  mesa para el modelo recurrente
# Construimos 2 secuencias con longitudes distintas

env_rnn_a = make_env(
    "table_realistic_default",
    shoe=["8", "6", "8", "10", "3", "K", "2", "10"],
    seed=11,
)
seq_a = [
    env_rnn_a.reset(),
    env_rnn_a.step("split"),
    env_rnn_a.step("double"),
    env_rnn_a.step("stand"),
]

env_rnn_b = make_env(
    "table_realistic_default",
    shoe=["10", "6", "7", "10", "10"],
    seed=12,
)
seq_b = [
    env_rnn_b.reset(),
    env_rnn_b.step("stand"),
]

print("len(seq_a):", len(seq_a))
print("len(seq_b):", len(seq_b))
print_response_summary(seq_a[0], "recurrent env A / first step")


len(seq_a): 4
len(seq_b): 2
recurrent env A / first step
observation profile: table_realistic_default
observation keys: ['current_bet', 'current_hand_cards', 'dealer_upcard', 'dealer_upcard_value', 'discard_summary', 'hand_context', 'insurance_context', 'mode', 'observed_cards_history', 'other_player_hands_visible', 'profile', 'temporal_context']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'resplit_aces_allowed', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 1, 1, 0]
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')


In [8]:
# instanciar RecurrentDoubleDQN en CPU

rnn_model = RecurrentDoubleDQN.from_profile(
    "table_realistic_default",
    recurrent_type="gru",
).to(device)
rnn_model.eval()

print(rnn_model)
print("state_dim:", rnn_model.state_dim)
print("num_actions:", rnn_model.num_actions)
print("device:", next(rnn_model.parameters()).device)


RecurrentDoubleDQN(
  (encoder): BlackjackObservationEncoder(
    (modules_by_name): ModuleDict(
      (hand): HandFeatureEncoder()
      (other_hands): OtherHandsEncoder()
      (hand_context): HandContextEncoder()
      (insurance): InsuranceContextEncoder()
      (bet): BetEncoder()
      (rules): RuleEncoder()
      (observed_history): ObservedCardsHistoryEncoder()
      (discard_summary): DiscardSummaryEncoder()
      (temporal): TemporalFeatureEncoder()
    )
  )
  (input_projection): Sequential(
    (0): Linear(in_features=1089, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
  )
  (recurrent_backbone): RecurrentBackbone(
    (recurrent): GRU(256, 256, batch_first=True)
  )
  (q_head): QHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=6, bias=True)
    )
  )
)
state_dim: 1089
num_actions: 6
device: cpu


In [9]:
# forward del RecurrentDoubleDQN con batch de secuencias

with torch.no_grad():
    rnn_out = rnn_model([seq_a, seq_b])

print("q_values shape:", tuple(rnn_out["q_values"].shape))
print("masked_q_values shape:", tuple(rnn_out["masked_q_values"].shape))
print("state_vector shape:", tuple(rnn_out["state_vector"].shape))
print("projected_state shape:", tuple(rnn_out["projected_state"].shape))
print("recurrent_output shape:", tuple(rnn_out["recurrent_output"].shape))
print("action_mask shape:", tuple(rnn_out["action_mask"].shape))
print("padding_mask shape:", tuple(rnn_out["padding_mask"].shape))

print("\nmetadata:")
pprint(rnn_out["metadata"])

print("\npadding_mask:")
print(rnn_out["padding_mask"])

print("\nq_values[0]:")
print(rnn_out["q_values"][0])

print("\nmasked_q_values[0]:")
print(rnn_out["masked_q_values"][0])


q_values shape: (2, 4, 6)
masked_q_values shape: (2, 4, 6)
state_vector shape: (2, 4, 1089)
projected_state shape: (2, 4, 256)
recurrent_output shape: (2, 4, 256)
action_mask shape: (2, 4, 6)
padding_mask shape: (2, 4)

metadata:
{'architecture': 'recurrent',
 'batch_size': 2,
 'encoder_profile': 'table_realistic_default',
 'items': [[{'module_dims': {'bet': 1,
                             'discard_summary': 144,
                             'hand': 185,
                             'hand_context': 5,
                             'insurance': 2,
                             'observed_history': 14,
                             'other_hands': 700,
                             'rules': 17,
                             'temporal': 21},
             'module_slices': {'bet': (892, 893),
                               'discard_summary': (924, 1068),
                               'hand': (0, 185),
                               'hand_context': (885, 890),
                               'insur

In [10]:
#  hidden state inicial explícito para el recurrente

hidden0 = rnn_model.init_hidden(batch_size=2, device=device)

if isinstance(hidden0, tuple):
    print("hidden is tuple")
    print([tuple(x.shape) for x in hidden0])
else:
    print("hidden is tensor")
    print(tuple(hidden0.shape))


hidden is tensor
(1, 2, 256)


In [11]:
# mesa unknown_progress para el dueling recurrent
# Este perfil oculta progreso estimado del shoe

start_state_a = StartStateConfig(
    mode="unknown_progress",
    min_burned_rounds=3,
    max_burned_rounds=3,
    hide_reshuffle_progress_from_observation=True,
)

env_duel_a = make_env(
    "table_realistic_unknown_progress",
    seed=21,
    start_state=start_state_a,
)
seq_duel_a = [
    env_duel_a.reset(),
    env_duel_a.step("stand"),
]

start_state_b = StartStateConfig(
    mode="unknown_progress",
    min_burned_rounds=2,
    max_burned_rounds=2,
    hide_reshuffle_progress_from_observation=True,
)

env_duel_b = make_env(
    "table_realistic_unknown_progress",
    seed=22,
    start_state=start_state_b,
)
seq_duel_b = [
    env_duel_b.reset(),
    env_duel_b.step("hit"),
]

print("len(seq_duel_a):", len(seq_duel_a))
print("len(seq_duel_b):", len(seq_duel_b))
print_response_summary(seq_duel_a[0], "dueling recurrent env / first step")

print("\ntemporal_context:")
pprint(seq_duel_a[0]["observation"]["temporal_context"])


len(seq_duel_a): 2
len(seq_duel_b): 2
dueling recurrent env / first step
observation profile: table_realistic_unknown_progress
observation keys: ['current_bet', 'current_hand_cards', 'dealer_upcard', 'dealer_upcard_value', 'discard_summary', 'hand_context', 'insurance_context', 'mode', 'observed_cards_history', 'other_player_hands_visible', 'profile', 'temporal_context']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'resplit_aces_allowed', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 0, 1, 0]
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')

temporal_context:
{'rounds_played_total': 1, 'shuffle_count': 0}


In [12]:
# instanciar DuelingRecurrentDoubleDQN en CPU

duel_model = DuelingRecurrentDoubleDQN.from_profile(
    "table_realistic_unknown_progress",
    recurrent_type="lstm",
).to(device)
duel_model.eval()

print(duel_model)
print("state_dim:", duel_model.state_dim)
print("num_actions:", duel_model.num_actions)
print("device:", next(duel_model.parameters()).device)


DuelingRecurrentDoubleDQN(
  (encoder): BlackjackObservationEncoder(
    (modules_by_name): ModuleDict(
      (hand): HandFeatureEncoder()
      (other_hands): OtherHandsEncoder()
      (hand_context): HandContextEncoder()
      (insurance): InsuranceContextEncoder()
      (bet): BetEncoder()
      (rules): RuleEncoder()
      (observed_history): ObservedCardsHistoryEncoder()
      (discard_summary): DiscardSummaryEncoder()
      (temporal): TemporalFeatureEncoder()
    )
  )
  (input_projection): Sequential(
    (0): Linear(in_features=1089, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
  )
  (recurrent_backbone): RecurrentBackbone(
    (recurrent): LSTM(256, 256, batch_first=True)
  )
  (dueling_head): DuelingQHead(
    (value_stream): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=1, bias=True)
    )
    (advantage_stream): Sequ

In [13]:
# forward batch del DuelingRecurrentDoubleDQN

with torch.no_grad():
    duel_out = duel_model([seq_duel_a, seq_duel_b])

print("q_values shape:", tuple(duel_out["q_values"].shape))
print("masked_q_values shape:", tuple(duel_out["masked_q_values"].shape))
print("state_value shape:", tuple(duel_out["state_value"].shape))
print("advantages shape:", tuple(duel_out["advantages"].shape))
print("state_vector shape:", tuple(duel_out["state_vector"].shape))
print("projected_state shape:", tuple(duel_out["projected_state"].shape))
print("recurrent_output shape:", tuple(duel_out["recurrent_output"].shape))
print("action_mask shape:", tuple(duel_out["action_mask"].shape))
print("padding_mask shape:", tuple(duel_out["padding_mask"].shape))

print("\nmetadata:")
pprint(duel_out["metadata"])


q_values shape: (2, 2, 6)
masked_q_values shape: (2, 2, 6)
state_value shape: (2, 2, 1)
advantages shape: (2, 2, 6)
state_vector shape: (2, 2, 1089)
projected_state shape: (2, 2, 256)
recurrent_output shape: (2, 2, 256)
action_mask shape: (2, 2, 6)
padding_mask shape: (2, 2)

metadata:
{'architecture': 'dueling_recurrent',
 'batch_size': 2,
 'encoder_profile': 'table_realistic_unknown_progress',
 'items': [[{'module_dims': {'bet': 1,
                             'discard_summary': 144,
                             'hand': 185,
                             'hand_context': 5,
                             'insurance': 2,
                             'observed_history': 14,
                             'other_hands': 700,
                             'rules': 17,
                             'temporal': 21},
             'module_slices': {'bet': (892, 893),
                               'discard_summary': (924, 1068),
                               'hand': (0, 185),
                      

In [14]:
# inspección del hidden state LSTM

hidden0_duel = duel_model.init_hidden(batch_size=2, device=device)

print("hidden state type:", type(hidden0_duel))
print("num tensors:", len(hidden0_duel))
print("h shape:", tuple(hidden0_duel[0].shape))
print("c shape:", tuple(hidden0_duel[1].shape))


hidden state type: <class 'tuple'>
num tensors: 2
h shape: (1, 2, 256)
c shape: (1, 2, 256)


In [15]:
# forward_step del DuelingRecurrentDoubleDQN
# Esto sirve para una sola observación + hidden state

single_step_env = make_env(
    "table_realistic_unknown_progress",
    seed=33,
    start_state=StartStateConfig(
        mode="unknown_progress",
        min_burned_rounds=1,
        max_burned_rounds=1,
        hide_reshuffle_progress_from_observation=True,
    ),
)

single_step_response = single_step_env.reset()

with torch.no_grad():
    step_hidden0 = duel_model.init_hidden(batch_size=1, device=device)
    step_out = duel_model.forward_step(single_step_response, hidden_state=step_hidden0)

print("q_values shape:", tuple(step_out["q_values"].shape))
print("masked_q_values shape:", tuple(step_out["masked_q_values"].shape))
print("state_value shape:", tuple(step_out["state_value"].shape))
print("advantages shape:", tuple(step_out["advantages"].shape))
print("state_vector shape:", tuple(step_out["state_vector"].shape))
print("projected_state shape:", tuple(step_out["projected_state"].shape))
print("recurrent_output shape:", tuple(step_out["recurrent_output"].shape))
print("action_mask shape:", tuple(step_out["action_mask"].shape))
print("padding_mask shape:", tuple(step_out["padding_mask"].shape))

print("\nq_values:")
print(step_out["q_values"])

print("\nmasked_q_values:")
print(step_out["masked_q_values"])


q_values shape: (1, 6)
masked_q_values shape: (1, 6)
state_value shape: (1, 1)
advantages shape: (1, 6)
state_vector shape: (1, 1089)
projected_state shape: (1, 256)
recurrent_output shape: (1, 256)
action_mask shape: (1, 6)
padding_mask shape: (1,)

q_values:
tensor([[-0.0147,  0.0809,  0.0247, -0.0320, -0.1002, -0.0465]])

masked_q_values:
tensor([[-1.4718e-02,  8.0863e-02,  2.4655e-02, -3.4028e+38, -1.0018e-01,
         -3.4028e+38]])


In [ ]:
# comparación rápida de los 3 modelos

print("FeedForward")
print("  state_dim:", ff_model.state_dim)
print("  q_values:", tuple(ff_out["q_values"].shape))

print("Recurrent")
print("  state_dim:", rnn_model.state_dim)
print("  q_values:", tuple(rnn_out["q_values"].shape))
print("  padding_mask:", tuple(rnn_out["padding_mask"].shape))

print("DuelingRecurrent")
print("  state_dim:", duel_model.state_dim)
print("  q_values:", tuple(duel_out["q_values"].shape))
print("  state_value:", tuple(duel_out["state_value"].shape))
print("  advantages:", tuple(duel_out["advantages"].shape))


FeedForward
  state_dim: 40
  q_values: (1, 6)
Recurrent
  state_dim: 1089
  q_values: (2, 4, 6)
  padding_mask: (2, 4)
DuelingRecurrent
  state_dim: 1089
  q_values: (2, 2, 6)
  state_value: (2, 2, 1)
  advantages: (2, 2, 6)
